# Phase‑1 Statistical Test Battery (cerruti)

This notebook runs a full battery of tests to evaluate whether the expert‑activation data (Phase‑1) shows structure beyond random noise.

It computes: entropy, normalized entropy, Gini coefficient, chi‑square + Cramer's V, mutual information (with permutation test), top‑K overlap permutation test, and a random‑split contr[...]

Configuration options below let you point the notebook at Phase‑1 or Phase‑2 JSON results.


In [ ]:
# Mount Google Drive in Colab and point the notebook to files on your Drive.
# Paste this cell near the top of the notebook before the configuration cell.
import os

# Default local paths (kept as fallback)
LOCAL_RESULT_JSON = '/content/lego_moe_expert_analysis.json'
LOCAL_OUTPUT_DIR = '/content/phase1_stat_tests_outputs'

try:
    # If running in Colab, this will succeed and mount your Drive
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_BASE = '/content/drive/MyDrive'  # change if you use a different Drive folder
    # Update these to match where you keep the results JSON on your Drive:
    RESULT_JSON = os.path.join(DRIVE_BASE, 'lego_moe_expert_analysis.json')
    OUTPUT_DIR = os.path.join(DRIVE_BASE, 'phase1_stat_tests_outputs')
    print('Using Drive paths:')
    print(' RESULT_JSON =', RESULT_JSON)
    print(' OUTPUT_DIR   =', OUTPUT_DIR)
except Exception:
    # Not in Colab — keep using local defaults (or set custom paths below)
    print('Not running in Colab; using local/default paths.')
    RESULT_JSON = LOCAL_RESULT_JSON
    OUTPUT_DIR = LOCAL_OUTPUT_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Configuration — edit these paths as needed before running
# For Phase-1 (local Colab run): /content/lego_moe_expert_analysis.json
# For Phase-2 (repo): data/phase2/runpod_second/results.json
RESULT_JSON = '/content/lego_moe_expert_analysis.json'  # primary path to analyze
RESULT_JSON_FALLBACK = 'data/phase2/runpod_second/results.json'  # alternate if above missing
OUTPUT_DIR = '/content/phase1_stat_tests_outputs'  # where outputs (plots/CSVs) will be written
N_PERMUTATIONS = 2000  # permutation iterations (increase for more precision)
TOP_K = 15  # for top-K overlap tests
SAMPLE_MI_MAX = 2_000_000  # cap expanded samples for mutual information to avoid OOM
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Outputs will be written to', OUTPUT_DIR)


In [ ]:
# Install any missing packages (run once)
import sys
!pip install -q scipy scikit-learn seaborn
print('Dependencies ensured')


In [ ]:
# Imports and helper functions
import json
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.metrics import mutual_info_score
sns.set(style='whitegrid')

def entropy(p):
    p = np.array(p, dtype=float)
    p = p[p>0]
    if p.size == 0:
        return 0.0
    return -np.sum(p * np.log2(p))

def gini(arr):
    a = np.array(arr, dtype=float)
    if a.sum()==0:
        return 0.0
    a = a.flatten()
    sorted_a = np.sort(a)
    n = len(a)
    index = np.arange(1, n+1)
    return (2*np.sum(index*sorted_a) - (n+1)*np.sum(sorted_a)) / (n*np.sum(sorted_a))

def cramers_v(contingency):
    chi2, p, dof, expected = chi2_contingency(contingency)
    n = contingency.sum()
    return np.sqrt(chi2 / (n * (min(contingency.shape)-1))), p

print('Helpers ready')


In [ ]:
# Load results JSON (try primary then fallback)
def load_results(primary, fallback=None):
    path = primary if os.path.exists(primary) else (fallback if fallback and os.path.exists(fallback) else None)
    if path is None:
        raise FileNotFoundError(f'Neither {primary} nor {fallback} exist. Update RESULT_JSON path and re-run.')
    print('Loading', path)
    with open(path, 'r') as f:
        data = json.load(f)
    per_q = data.get('per_question_results') or data.get('per_question_results') or data.get('per_question') or data.get('per_question_results') or data.get('per_question_results') or data.g[...]
    # permissive: if top-level is dict of qid -> {...} that's per_q. If it's a list, convert.
    if isinstance(per_q, list):
        per_q = {str(i): item for i,item in enumerate(per_q)}
    return per_q

per_question = load_results(RESULT_JSON, RESULT_JSON_FALLBACK)
print('Loaded', len(per_question), 'questions')


In [ ]:
# Build per-question expert vectors and category lists
questions_by_category = defaultdict(list)
expert_set = set()
for qid, q in per_question.items():
    cat = q.get('category')
    if cat is None:
        continue
    questions_by_category[cat].append(qid)
    freq = q.get('expert_frequency') or q.get('image_expert_frequency') or q.get('text_expert_frequency') or {}
    for eid in freq.keys():
        try:
            expert_set.add(int(eid))
        except Exception:
            expert_set.add(eid)
expert_ids = sorted(expert_set)
print('Found', len(expert_ids), 'experts across', len(questions_by_category), 'categories')

# function to turn expert_frequency dict into vector aligned with expert_ids
def freq_dict_to_vec(freq_dict):
    vec = np.zeros(len(expert_ids), dtype=float)
    for k,v in (freq_dict or {}).items():
        try:
            idx = expert_ids.index(int(k))
        except Exception:
            try:
                idx = expert_ids.index(k)
            except Exception:
                continue
        vec[idx] = int(v)
    return vec

# build per-question vectors (sparse-friendly)
per_q_vecs = {}
for qid, q in per_question.items():
    freq = q.get('expert_frequency') or q.get('image_expert_frequency') or {}
    per_q_vecs[qid] = freq_dict_to_vec(freq)

print('Built per-question vectors for', len(per_q_vecs), 'questions')


In [ ]:
# Aggregate by category and compute descriptive metrics
category_vecs = {}
for cat, qlist in questions_by_category.items():
    agg = np.sum([per_q_vecs[qid] for qid in qlist], axis=0) if qlist else np.zeros(len(expert_ids))
    category_vecs[cat] = agg

# descriptive table
rows = []
for cat, vec in category_vecs.items():
    total = vec.sum()
    if total>0:
        p = vec/total
        H = entropy(p)
        H_norm = H / math.log2(len(vec)) if len(vec)>1 else 0.0
        G = gini(vec)
    else:
        H=H_norm=G=0.0
    rows.append({'category':cat,'total_activations':int(total),'entropy':float(H),'norm_entropy':float(H_norm),'gini':float(G)})
df_desc = pd.DataFrame(rows).sort_values('total_activations',ascending=False)
df_desc.to_csv(os.path.join(OUTPUT_DIR,'category_descriptive.csv'), index=False)
print('Saved category descriptive to', os.path.join(OUTPUT_DIR,'category_descriptive.csv'))
df_desc


In [ ]:
# Chi-square / Cramer's V across expert x category contingency
# Build contingency matrix: rows=experts, cols=categories
cats = sorted(category_vecs.keys())
cont = np.stack([category_vecs[c] for c in cats], axis=1)  # shape (num_experts, num_cats)
cv, p_chi = cramers_v(cont)
print('Cramer\'s V:', cv, 'Chi2 p-value:', p_chi)
# save contingency head for inspection
pd.DataFrame(cont, index=[str(e) for e in expert_ids], columns=cats).head(10).to_csv(os.path.join(OUTPUT_DIR,'contingency_head.csv'))


In [ ]:
# Mutual Information between expert id and category (with permutation test)
# We expand counts into arrays but cap the total expanded size to SAMPLE_MI_MAX for memory safety
expert_labels = []
category_labels = []
total_counts = sum([v.sum() for v in category_vecs.values()])
ratio = min(1.0, SAMPLE_MI_MAX / float(total_counts) if total_counts>0 else 1.0)
for cat, vec in category_vecs.items():
    for idx, cnt in enumerate(vec):
        if cnt<=0:
            continue
        # scaled count to avoid extreme expansions
        k = max(1, int(round(cnt * ratio)))
        expert_labels += [idx]*k
        category_labels += [cat]*k

expert_labels = np.array(expert_labels)
category_labels = np.array(category_labels)
print('Expanded sample size for MI:', len(expert_labels))
if len(expert_labels) > 0:
    mi_obs = mutual_info_score(category_labels, expert_labels)
    # permutation test
    perms = 500
    greater = 0
    rng = np.random.RandomState(0)
    for i in range(perms):
        perm = rng.permutation(category_labels)
        mi_p = mutual_info_score(perm, expert_labels)
        if mi_p >= mi_obs:
            greater += 1
    p_mi = (greater+1)/(perms+1)
    print('MI obs:', mi_obs, 'p≈', p_mi, ' (',perms,'permutations)')
    with open(os.path.join(OUTPUT_DIR,'mi_results.txt'),'w') as of:
        of.write(f'MI_obs={mi_obs}, p~={p_mi}, perms={perms}
')
else:
    print('No data for MI computation')


In [ ]:
# Top-K overlap permutation test across categories
from itertools import combinations
all_experts = list(range(len(expert_ids)))
topk = TOP_K
top_sets = {}
for cat, vec in category_vecs.items():
    top_idx = np.argsort(vec)[::-1][:topk]
    top_sets[cat] = set([expert_ids[i] for i in top_idx])

observed_pairwise = {}
for a,b in combinations(list(top_sets.keys()),2):
    observed_pairwise[(a,b)] = len(top_sets[a] & top_sets[b])
print('Observed top-K intersections:', observed_pairwise)

# permutation: randomly sample top-K sets from expert pool to produce null distribution
trials = N_PERMUTATIONS
rng = np.random.RandomState(0)
perm_pvals = {}
for a,b in observed_pairwise.keys():
    obs = observed_pairwise[(a,b)]
    cnt_ge = 0
    for t in range(trials):
        sa = set(rng.choice(expert_ids, size=topk, replace=False))
        sb = set(rng.choice(expert_ids, size=topk, replace=False))
        if len(sa & sb) >= obs:
            cnt_ge += 1
    pval = (cnt_ge+1)/(trials+1)
    perm_pvals[(a,b)] = pval
print('Permutation p-values for top-K intersections:', perm_pvals)
pd.DataFrame([{'pair':f'{a}__{b}','observed':observed_pairwise[(a,b)],'pval':perm_pvals[(a,b)]} for (a,b) in observed_pairwise]).to_csv(os.path.join(OUTPUT_DIR,'topk_overlap_pvals.csv'), ind[...]
print('Saved top-K p-values to', os.path.join(OUTPUT_DIR,'topk_overlap_pvals.csv'))


In [ ]:
# Random-split control and permutation histogram (label-shuffle) for a pair of categories
# Defaults: compare the two largest categories by activations
cats_sorted = sorted([(cat,category_vecs[cat].sum()) for cat in category_vecs.keys()], key=lambda x: x[1], reverse=True)
if len(cats_sorted) < 2:
    print('Not enough categories for a pairwise comparison')
else:
    # pick two categories (you can change these manually at the top)
    if 'CATEGORY_A' in globals():
        catA = CATEGORY_A if CATEGORY_A in category_vecs else cats_sorted[0][0]
    else:
        catA = cats_sorted[0][0]
    if 'CATEGORY_B' in globals():
        catB = CATEGORY_B if CATEGORY_B in category_vecs else cats_sorted[1][0]
    else:
        catB = cats_sorted[1][0]
    print('Comparing', catA, 'vs', catB)
    qidsA = questions_by_category[catA]
    qidsB = questions_by_category[catB]
    nA, nB = len(qidsA), len(qidsB)
    combined = list(qidsA) + list(qidsB)
    # observed metric: L2 norm between aggregated expert vectors
    aggA = category_vecs[catA]
    aggB = category_vecs[catB]
    obs_diff = np.linalg.norm(aggA - aggB)
    print('Observed L2 norm:', obs_diff)
    # permutation: random split of combined questions into groups of sizes nA/nB
    rng = np.random.RandomState(0)
    metrics = []
    trials = N_PERMUTATIONS
    for t in range(trials):
        perm = rng.permutation(combined)
        randA = perm[:nA]
        randB = perm[nA:nA+nB]
        agg_randA = np.sum([per_q_vecs[qid] for qid in randA], axis=0)
        agg_randB = np.sum([per_q_vecs[qid] for qid in randB], axis=0)
        metrics.append(np.linalg.norm(agg_randA - agg_randB))
        if (t+1) % 200 == 0:
            print('perm done', t+1)
    metrics = np.array(metrics)
    p_perm = (np.sum(metrics >= obs_diff) + 1) / (len(metrics) + 1)
    print('Random-split permutation p-value:', p_perm)
    # save metrics and histogram
    pd.DataFrame({'l2_norm':metrics}).to_csv(os.path.join(OUTPUT_DIR,f'permutation_metrics_{catA}_vs_{catB}.csv'), index=False)
    plt.figure(figsize=(8,4))
    plt.hist(metrics, bins=50, alpha=0.8)
    plt.axvline(obs_diff, color='red', linewidth=2, label='observed')
    plt.xlabel('L2 norm of diff (random split)')
    plt.ylabel('count')
    plt.title(f'Permutation distribution: {catA} vs {catB}')
    plt.legend()
    out_hist = os.path.join(OUTPUT_DIR,f'perm_hist_{catA}_vs_{catB}.png')
    plt.tight_layout()
    plt.savefig(out_hist, dpi=200, bbox_inches='tight')
    print('Saved', out_hist)
    plt.show()


# Summary and next steps

The notebook saved descriptive tables, permutation metrics, and plots to the OUTPUT_DIR.

Next actions you might take:
- Increase N_PERMUTATIONS for tighter p-values (warning: linear cost).
- Use per-question normalization (divide each v_q by token count) if questions vary in length.
- Run the same battery on Sprint‑2 `data/phase2/runpod_second/results.json` to compare results.

If you want, I can also commit a version of this notebook that writes outputs directly to Google Drive (so you don't need to manually copy).